Migrated from Databricks to Neuron (auto-port).

In [ ]:
import os
import sys
from pathlib import Path

for _candidate in (
    "/home/jovyan/work/default/rashmi-ranjan/lib",
    str(Path.cwd().parent / "lib"),
    str(Path.cwd() / "lib"),
):
    if _candidate not in sys.path and Path(_candidate).is_dir():
        sys.path.insert(0, _candidate)

from neuron_runtime import (
    GoogleSheetReader,
    SlackReportsTrigger,
    get_sql_df,
    load_service_account,
    load_slack_token,
    make_unique_columns,
)

service_account = load_service_account()
slack_token = load_slack_token()

In [ ]:
# DBTITLE 1,Cell 2
import pandas as pd

query = """
WITH base_calls AS (
    SELECT
        CAST(to_date(to_timestamp(t.created_at)) AS DATE) AS call_date,
        t.created_at AS call_timestamp,
        t.uuid AS ticket_uuid,
        cb.navi_customer_id
    FROM awsdatacatalog.crm_tickets.tickets t
    LEFT JOIN awsdatacatalog.gi_io_service.callback_requests cb
        ON t.uuid = cb.external_ticket_id
        AND cb.insurance_type = 'MOTOR'
    WHERE t.ticket_group = 'GI_MOTOR_CALLING'
      AND t.ticket_state = 'RESOLVED'
      AND get_json_object(CAST(t.meta_data AS STRING), '$.recordingUrl') IS NOT NULL
      AND CAST(to_date(to_timestamp(t.created_at)) AS DATE)
          BETWEEN date_sub(current_date(), 90)
              AND date_sub(current_date(), 1)
),

sales_data_dedup AS (
    SELECT
        customer_id AS navi_customer_id,
        CAST(sale_date AS DATE) AS sale_date
    FROM (
        SELECT
            customer_id,
            sale_date,
            ROW_NUMBER() OVER (
                PARTITION BY customer_id
                ORDER BY sale_date DESC
            ) AS rn
        FROM awsdatacatalog.corp_analytics.gi_jarvis_sale_master_motor
        WHERE customer_id IS NOT NULL
          AND is_jarvis = 1
    )
    WHERE rn = 1
),

final_data AS (
    SELECT
        CASE
            WHEN sd.navi_customer_id IS NOT NULL THEN sd.sale_date
            ELSE bc.call_date
        END AS call_date,
        CASE
            WHEN sd.navi_customer_id IS NOT NULL THEN 'SALE_AUDIT'
            ELSE 'NON_SALE_AUDIT'
        END AS audit_type,
        bc.navi_customer_id
    FROM base_calls bc
    LEFT JOIN sales_data_dedup sd
        ON bc.navi_customer_id = sd.navi_customer_id
        AND bc.call_date <= sd.sale_date
),

aggregated AS (
    SELECT
        call_date,
        CASE
            WHEN dayofweek_iso(current_date()) = 1
                 AND dayofweek_iso(call_date) IN (6, 7)
                THEN date_add(current_date(), 1)
            ELSE date_add(call_date, 1)
        END AS audit_date,
        COUNT(CASE WHEN audit_type = 'SALE_AUDIT' THEN 1 END) AS sale_conversations_count,
        COUNT(CASE WHEN audit_type = 'NON_SALE_AUDIT' THEN 1 END) AS non_sale_conversations_count,
        COUNT(DISTINCT CASE WHEN audit_type = 'SALE_AUDIT' THEN navi_customer_id END) AS sale_customer_id_count
    FROM final_data
    WHERE call_date < current_date()
    GROUP BY
        call_date,
        CASE
            WHEN dayofweek_iso(current_date()) = 1
                 AND dayofweek_iso(call_date) IN (6, 7)
                THEN date_add(current_date(), 1)
            ELSE date_add(call_date, 1)
        END
)

SELECT *
FROM aggregated
ORDER BY call_date DESC
"""

result = get_sql_df(query)
df, code = result if isinstance(result, tuple) else (result, None)

if df is not None and not df.empty:
    display(df)
else:
    print("No rows returned.", f"Code={code}" if code is not None else "")

In [ ]:
# Python cell

In [ ]:
import os
import shutil

if not os.path.exists("data"):
    os.makedirs("data")

gsheet_url = "https://docs.google.com/spreadsheets/d/1pV1Rx7fkuJagQRulyPAI9GCdJ5NbOYi7mc94XNR8E28/edit?gid=1783467241#gid=1783467241"
worksheet_name = "Connected Call Count Jarvis"
gs_reader = GoogleSheetReader(gsheet_url, service_account)

gs_reader.clear_sheets(
    worksheet_name,
    start_row=1,
    start_column=1
)

combined_df = pd.concat([df], ignore_index=True) 

gs_reader.set_df_in_sheets( 
worksheet_name, 
combined_df, 
include_headers=True, 
row=1, 
col=1
)